[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/23_cross_attention.ipynb)

# 🟠 Medium: Multi-Head Cross-Attention

Implement **multi-head cross-attention** (encoder-decoder attention).

### Signature
```python
class MultiHeadCrossAttention(nn.Module):
    def __init__(self, d_model: int, num_heads: int): ...
    def forward(self, x_q: Tensor, x_kv: Tensor) -> Tensor:
        # x_q: (B, S_q, D) — decoder queries
        # x_kv: (B, S_kv, D) — encoder keys/values
```

### Key Differences from Self-Attention
- Q comes from the decoder, K and V come from the encoder
- No causal mask (all encoder positions visible)

In [1]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 1.7 MB/s eta 0:00:00


In [1]:
import torch
import torch.nn as nn
import math

In [4]:
class MultiHeadCrossAttention(nn.Module):
    def __init__(self, d_model: int, num_heads: int):
        super().__init__()

        self.d_model = d_model
        self.num_heads = num_heads
        assert d_model % num_heads == 0

        self.head_dim = d_model // num_heads

        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)

    def forward(self, xq, xv):
        """
        xq: (B, S_q, d_model)
        xv: (B, S_k, d_model)

        return: (B, S_q, d_model)
        """

        B = xq.shape[0]
        S_q = xq.shape[1]
        S_k = xv.shape[1]

        Q = self.W_q(xq).reshape(B, S_q, self.num_heads, self.head_dim).permute(0, 2, 1, 3) #(B, num_heads, S_q, head_dim)

        K = self.W_k(xv).reshape(B, S_k, self.num_heads, self.head_dim).permute(0, 2, 1, 3) #(B, num_heads, S_k, head_dim)

        V = self.W_v(xv).reshape(B, S_k, self.num_heads, self.head_dim).permute(0, 2, 1, 3) #(B, num_heads, S_k, head_dim)


        d_k = K.shape[-1]
        att_weights = torch.softmax(Q @ K.permute(0, 1, 3, 2) / (d_k ** 0.5), dim = -1) #(B, num_heads, S_q, S_k)

        hidden = att_weights @ V #(B, num_heads, S_q, head_dim)
        hidden = hidden.permute(0, 2, 1, 3).reshape(B, S_q, self.d_model) #(B, S_q, head_dim * num_heads = d_model)

        out = self.W_o(hidden) #(B, S_q, d_model)
        return out


In [5]:
# 🧪 Debug
attn = MultiHeadCrossAttention(64, 4)
x_q = torch.randn(2, 6, 64)
x_kv = torch.randn(2, 10, 64)
print('Output:', attn(x_q, x_kv).shape)

Output: torch.Size([2, 6, 64])


In [7]:
# ✅ SUBMIT
from torch_judge import check
check('cross_attention')


🧪 Testing: Multi-Head Cross-Attention (Medium)
──────────────────────────────────────────────────
  ✅ [1/4] Output shape (2.3ms)
  ✅ [2/4] Q and KV different lengths (0.8ms)
  ✅ [3/4] No causal mask — all KV affects all Q (1.5ms)
  ✅ [4/4] Gradient flow (0.7ms)
──────────────────────────────────────────────────
  🎉 All 4 tests passed! (5.3ms total)
  Progress saved. Run status() to see your dashboard.

